# SegFormer Training and Evaluation on LIVECell Dataset
This notebook performs training, validation, and evaluation of a SegFormer model for multi-class semantic segmentation on the LIVECell dataset.

In [ ]:
# Setup: Install dependencies
!pip install transformers timm opencv-python scikit-learn matplotlib albumentations onnx onnxruntime

## Step 1: Load Config and Utilities

In [ ]:
from config import Config, setup_logger
from dataset import LIVECellSegDataset
from model import SegFormerModelWrapper
from trainer import SegFormerTrainer
from evaluation import evaluate_on_test_set
from utils import visualize_triplet_images
import os
import torch

cfg = Config()
logger = setup_logger()
device = torch.device(cfg.DEVICE)

## Step 2: Initialize Model and Trainer

In [ ]:
model_wrapper = SegFormerModelWrapper(cfg.MODEL_SAVE_PATH, device, cfg.MODEL_TYPE, cfg.NUM_CLASSES)
optimizer = torch.optim.AdamW(model_wrapper.model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.5)
trainer = SegFormerTrainer(model_wrapper, optimizer, scheduler, cfg, logger)

## Step 3: Load Dataset

In [ ]:
train_dataset = LIVECellSegDataset(cfg.TRAIN_IMG_DIR, cfg.TRAIN_MASK_DIR, model_wrapper.processor)
val_dataset = LIVECellSegDataset(cfg.VAL_IMG_DIR, cfg.VAL_MASK_DIR, model_wrapper.processor)
test_dataset = LIVECellSegDataset(cfg.TEST_IMG_DIR, cfg.TEST_MASK_DIR, model_wrapper.processor)

In [5]:
from torch.utils.data import DataLoader
train_loader = DataLoader(train_dataset, batch_size=cfg.BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=cfg.BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=cfg.BATCH_SIZE)

## Step 4: Train the Model

In [ ]:
trainer.train(train_loader, val_loader)

## Step 5: Evaluate on Test Set

In [ ]:
evaluate_on_test_set(model_path=cfg.MODEL_SAVE_PATH, test_img_dir=cfg.TEST_IMG_DIR, test_mask_dir=cfg.TEST_MASK_DIR, device=device)